# 03 · Plata → Oro: universo, piloto y top 50

**Componente B · Sprint 2** · Fase CRISP-DM: *modelado* · Regla aprobada en la reunión del 04-nov-2026 (RN-02)

| Regla | Detalle |
| --- | --- |
| Grupos | Núcleo tecnológico (J61, J62, J63, J582) + secciones G, C, M y K |
| Universo | Tecnológicas de todos los tamaños + medianas y grandes de G, C, M y K |
| Piloto | 300 tecnológicas medianas y grandes + top 50 por ventas de G, C, M y K |
| Holdings en K | Fuera del top 50 (no ofrecen productos al público); siguen en el universo |
| Excluidas | Filas con año 2025 = 0 |

| Entrada | Salida (oro) |
| --- | --- |
| `plata/ranking_2025.parquet` | `oro/universo_2025.parquet`, `oro/piloto_2025.csv`, `oro/top50_<grupo>.csv` |

In [1]:
from pathlib import Path

ETL = Path.cwd().resolve()
if ETL.name == "notebooks":
    ETL = ETL.parent
DATOS = ETL / "datos"
CRUDO = DATOS / "crudo"      # bronce: datos tal como llegan, sin modificar
SCVS = CRUDO / "2025"        # descargas oficiales de la SCVS
WEB = CRUDO / "web"          # páginas capturadas de las webs oficiales
PLATA = DATOS / "plata"      # datos limpios y validados
ORO = DATOS / "oro"          # datos listos para publicar
for carpeta in (WEB, PLATA, ORO):
    carpeta.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd

plata = pd.read_parquet(PLATA / "ranking_2025.parquet")
len(plata)

154687

## 1. Grupo de cada empresa

In [3]:
PREFIJOS_TEC = ("J61", "J62", "J63", "J582")

def grupo_de(ciiu: str):
    if ciiu.startswith(PREFIJOS_TEC):
        return "TEC"
    if ciiu[0] in ("G", "C", "M", "K"):
        return ciiu[0]
    return None

plata["grupo"] = plata["ciiu"].map(grupo_de)
plata["es_holding"] = plata["ciiu"].str.startswith("K642") | plata["nombre"].str.contains("HOLDING", regex=False)
pd.crosstab(plata["grupo"], plata["tamano_derivado"], margins=True)

tamano_derivado,grande,mediana,micro_pequena,All
grupo,,,,
C,710,1039,8218,9967
G,1763,3447,31985,37195
K,187,330,3375,3892
M,108,546,21260,21914
TEC,63,237,5599,5899
All,2831,5599,70437,78867


## 2. Universo

In [4]:
valida = plata[plata["grupo"].notna() & ~plata["anio_cero"]]
universo = valida[(valida["grupo"] == "TEC") | (valida["tamano_derivado"] != "micro_pequena")].copy()
universo["rank_grupo"] = universo.groupby("grupo")["ventas"].rank(ascending=False, method="first").astype(int)
universo = universo.sort_values(["grupo", "rank_grupo"]).reset_index(drop=True)
universo["grupo"].value_counts().to_frame().assign(total=lambda t: t.sum().iloc[0])

,count,total
grupo,,
TEC,5898,14027
G,5210,14027
C,1748,14027
M,654,14027
K,517,14027


## 3. Top 50 y piloto

In [5]:
TOP_N = 50
tops = {}
for g in ["TEC", "G", "C", "M", "K"]:
    candidatos = universo[universo["grupo"] == g]
    if g == "K":
        candidatos = candidatos[~candidatos["es_holding"]]
    if g == "TEC":
        candidatos = candidatos[candidatos["tamano_derivado"] != "micro_pequena"]
    tops[g] = candidatos.nsmallest(TOP_N, "rank_grupo")

tec_mg = universo[(universo["grupo"] == "TEC") & (universo["tamano_derivado"] != "micro_pequena")]
piloto = pd.concat(
    [tec_mg.assign(criterio_piloto="tecnologica_mediana_grande")]
    + [tops[g].assign(criterio_piloto=f"top50_{g.lower()}") for g in ["G", "C", "M", "K"]]
)
piloto["en_top50_tec"] = piloto["expediente"].isin(tops["TEC"]["expediente"])
piloto["criterio_piloto"].value_counts()

criterio_piloto
tecnologica_mediana_grande    300
top50_g                        50
top50_c                        50
top50_m                        50
top50_k                        50
Name: count, dtype: int64

## 4. Validación cruzada con la selección de A

Se compara contra los archivos que generó el Componente A el 09-nov-2026 (`datos/universo/2025/`). Deben coincidir; si no, se explica la diferencia.

In [6]:
REF = DATOS / "universo" / "2025"
ref_universo = pd.read_csv(REF / "universo_2025.csv", usecols=["expediente"])
ref_piloto = pd.read_csv(REF / "piloto_500.csv", usecols=["expediente"])

def comparar(mio, ref, nombre):
    a, b = set(mio["expediente"]), set(ref["expediente"])
    return {"conjunto": nombre, "B": len(a), "A": len(b), "solo_B": len(a - b), "solo_A": len(b - a)}

comparacion = pd.DataFrame([comparar(universo, ref_universo, "universo"), comparar(piloto, ref_piloto, "piloto")])
comparacion

,conjunto,B,A,solo_B,solo_A
0,universo,14027,14029,0,2
1,piloto,500,500,0,0


In [7]:
solo_b = set(universo["expediente"]) - set(ref_universo["expediente"])
solo_a = set(ref_universo["expediente"]) - set(universo["expediente"])
diferencias = plata[plata["expediente"].isin(solo_a | solo_b)][
    ["expediente", "nombre", "grupo", "anio_2025", "ventas", "tamano_derivado"]
]
diferencias.assign(esta_en=diferencias["expediente"].map(lambda e: "solo B" if e in solo_b else "solo A"))

,expediente,nombre,grupo,anio_2025,ventas,tamano_derivado,esta_en
2,305653,REFRIGERACION INDUSTRIAL CORNEJO REFRICOR-EC C...,C,0,1014214.76,mediana,solo A
22,731722,PAYJOY ECUADOR S.A.S.,TEC,0,244690.83,micro_pequena,solo A


## 5. Guardar oro

In [8]:
COLUMNAS = [
    "expediente", "ruc", "nombre", "grupo", "ciiu", "ciiu_descripcion", "seccion_ciiu", "division_ciiu",
    "tamano_derivado", "tipo_compania", "region", "provincia", "ciudad", "empleados", "activo",
    "patrimonio", "ventas", "utilidad_neta", "ingreso_total", "es_holding", "alerta_empleados",
    "nueva_en_ranking", "rank_grupo",
]
universo[COLUMNAS].to_parquet(ORO / "universo_2025.parquet", index=False)
piloto[COLUMNAS + ["criterio_piloto", "en_top50_tec"]].to_csv(ORO / "piloto_2025.csv", index=False)
for g, t in tops.items():
    t[COLUMNAS].to_csv(ORO / f"top50_{g.lower()}.csv", index=False)
sorted(p.name for p in ORO.iterdir())

['piloto_2025.csv',
 'top50_c.csv',
 'top50_g.csv',
 'top50_k.csv',
 'top50_m.csv',
 'top50_tec.csv',
 'universo_2025.parquet']